In [15]:
# Importações e Configuração Inicial

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

# Configurações visuais
sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14

In [18]:
# Carregando a FatoTasks com automatização de separador (devido inconsistências da integração com Devops)
import csv

with open("TabelaFatoTasks.csv", "r", encoding="utf-8", errors="ignore") as f:
    dialect = csv.Sniffer().sniff(f.read(2048))
    print("Separador detectado:", dialect.delimiter)

Separador detectado: ;


In [19]:
df = pd.read_csv(arquivo, sep=";", encoding="utf-8")

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29997 entries, 0 to 29996
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Area Path                 29997 non-null  str  
 1   Assigned To               19681 non-null  str  
 2   Ano                       29997 non-null  int64
 3   Trimestre                 29997 non-null  str  
 4   Mês                       29997 non-null  str  
 5   Dia                       29997 non-null  int64
 6   Is Current                29997 non-null  bool 
 7   Iteration Path            29997 non-null  str  
 8   State                     29997 non-null  str  
 9   Title                     29997 non-null  str  
 10  Contagem de Work Item Id  29974 non-null  str  
 11  Work Item Type            29974 non-null  str  
 12  Unnamed: 12               54 non-null     str  
dtypes: bool(1), int64(2), str(10)
memory usage: 2.8 MB


In [21]:
for i, col in enumerate(df.columns, start=1):
    print(i, "->", col)

1 -> Area Path
2 -> Assigned To
3 -> Ano
4 -> Trimestre
5 -> Mês
6 -> Dia
7 -> Is Current
8 -> Iteration Path
9 -> State
10 -> Title
11 -> Contagem de Work Item Id
12 -> Work Item Type
13 -> Unnamed: 12


In [22]:
# Padronizar nomes de colunas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("ã", "a")
    .str.replace("á", "a")
    .str.replace("é", "e")
    .str.replace("í", "i")
    .str.replace("ó", "o")
    .str.replace("ú", "u")
    .str.replace("ç", "c")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

df.columns

Index(['area_path', 'assigned_to', 'ano', 'trimestre', 'ms', 'dia',
       'is_current', 'iteration_path', 'state', 'title',
       'contagem_de_work_item_id', 'work_item_type', 'unnamed_12'],
      dtype='str')

In [23]:
# Detectando automaticamente as colunas de ano, mes e dia
col_ano = [c for c in df.columns if "ano" in c][0]
col_mes = [c for c in df.columns if "mes" in c][0]
col_dia = [c for c in df.columns if "dia" in c][0]

col_ano, col_mes, col_dia

('ano', 'trimestre', 'dia')

In [24]:
df["data"] = pd.to_datetime(
    df[[col_ano, col_mes, col_dia]].astype(str).agg("-".join, axis=1),
    errors="coerce"
)

df[["data"]].head()

C:\Users\jesse\AppData\Local\Temp\ipykernel_14320\262989324.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["data"] = pd.to_datetime(


,data
0,NaT
1,NaT
2,NaT
3,NaT
4,NaT


In [25]:
df.isna().sum().sort_values(ascending=False)

data                        29997
unnamed_12                  29943
assigned_to                 10316
contagem_de_work_item_id       23
work_item_type                 23
area_path                       0
dia                             0
ms                              0
trimestre                       0
ano                             0
title                           0
state                           0
iteration_path                  0
is_current                      0
dtype: int64